In [2]:
# ============================================
# CIFAR-10 Image Classification using CNN
# Dataset: CIFAR-10-images-master (from Kaggle, extracted from archive.zip)
# Structure: train/<class_name>/*.jpg, test/<class_name>/*.jpg
# ============================================

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

# ----------------------------------------------------
# STEP 1: Set paths (update if your folder is elsewhere)
# ----------------------------------------------------
# Using a smaller balanced subset: 350 train images/class (3500 total)
# and 70 test images/class (700 total) for faster experimentation.

TRAIN_DIR = "/content/drive/MyDrive/Colab Notebooks/cifar10_subset/train"
TEST_DIR = "/content/drive/MyDrive/Colab Notebooks/cifar10_subset/test"

IMG_SIZE = (32, 32)
BATCH_SIZE = 32

# ----------------------------------------------------
# STEP 2: Load Dataset from directories
# ----------------------------------------------------

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

# ----------------------------------------------------
# STEP 3: Visualize Sample Images
# ----------------------------------------------------

plt.figure(figsize=(10, 4))
for images, labels in train_ds.take(1):
    for i in range(10):
        plt.subplot(2, 5, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]])
        plt.axis('off')
plt.tight_layout()
plt.show()

# ----------------------------------------------------
# STEP 4: Preprocess - Normalize + Augment
# ----------------------------------------------------

normalization_layer = layers.Rescaling(1./255)

data_augmentation = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
])

AUTOTUNE = tf.data.AUTOTUNE

def prepare(ds, training=False):
    ds = ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.cache().prefetch(buffer_size=AUTOTUNE)

train_ds_prepped = prepare(train_ds, training=True)
test_ds_prepped = prepare(test_ds, training=False)

# ----------------------------------------------------
# STEP 5: Build the CNN Model
# ----------------------------------------------------

model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.summary()

# ----------------------------------------------------
# STEP 6: Compile the Model
# ----------------------------------------------------

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ----------------------------------------------------
# STEP 7: Train the Model
# ----------------------------------------------------

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

history = model.fit(
    train_ds_prepped,
    epochs=30,
    validation_data=test_ds_prepped,
    callbacks=[early_stop]
)

# ----------------------------------------------------
# STEP 8: Evaluate the Model
# ----------------------------------------------------

test_loss, test_acc = model.evaluate(test_ds_prepped)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")

# ----------------------------------------------------
# STEP 9: Plot Training History
# ----------------------------------------------------

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.show()

# ----------------------------------------------------
# STEP 10: Confusion Matrix
# ----------------------------------------------------

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true = np.concatenate([y.numpy() for x, y in test_ds_prepped], axis=0)
y_pred_probs = model.predict(test_ds_prepped)
y_pred = np.argmax(y_pred_probs, axis=1)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

# ----------------------------------------------------
# STEP 11: Save the Model
# ----------------------------------------------------

model.save('cifar10_cnn_model.keras')
print("Model saved successfully!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/cifar10_subset/train'